# Exercício 24 — Pesquisa, enquete e representatividade

**Pergunta:** por que uma pesquisa aleatória pequena pode acertar mais que uma enquete gigante, e o que a ponderação consegue (ou não) corrigir?

**Entrega:** pasta `24-exercicio-amostragem/` no seu repositório privado, com este notebook executado e um `README.md` curto contendo instruções de execução, fonte dos dados (simulados + a ficha técnica do PesqEle que você usou), limitações e a **declaração de uso de IA** (ferramenta, objetivo, o que você alterou depois de conferir).

**Como executar:** abra no Colab, ou local com `uv venv .venv` e `uv pip install -r requirements.txt`. Rode as células de cima para baixo. Em cada parte, o que você precisa escrever está marcado com `# SEU CÓDIGO AQUI`.

**Individualização:** o seu eleitorado é gerado a partir de uma **semente**. Use os quatro últimos dígitos da sua matrícula. Seus números serão diferentes dos colegas, mas o raciocínio é o mesmo.

## 0. Gerar o seu eleitorado fictício (já pronto, só rode)

Troque `semente` pelo número da sua matrícula (quatro dígitos).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

semente = 2083   # TROQUE: quatro últimos dígitos da sua matrícula
rng = np.random.default_rng(semente)

N = 200_000
faixas = np.array(["16-34", "35-59", "60+"])
idx = rng.choice(3, size=N, p=[0.34, 0.44, 0.22])
apoio_por_faixa = rng.uniform(0.25, 0.60, size=3)     # muda com a sua semente
apoia = rng.random(N) < apoio_por_faixa[idx]

eleitorado = pd.DataFrame({"faixa": faixas[idx], "apoia": apoia})
print(eleitorado.shape)
eleitorado.head()

(200000, 2)


,faixa,apoia
0,60+,False
1,35-59,False
2,35-59,True
3,16-34,True
4,16-34,False


## 1. A verdade (carregar → inspecionar)

Calcule a proporção do eleitorado que apoia o candidato (**o parâmetro**) e a composição por faixa etária. Guarde em `verdade` e `composicao`. Escreva em uma linha o que é "parâmetro" para você.

In [2]:
# SEU CÓDIGO AQUI

verdade = eleitorado["apoia"].mean() 
composicao = eleitorado["faixa"].value_counts(normalize=True).sort_index()
print(f"Verdade: {verdade:.0%} do eleitorado apoia o candidato.")

# Parâmetro é o valor verdadeiro de algo na população inteira, que normalmente não é conhecido e tenta se estimar com uma amostra


Verdade: 39% do eleitorado apoia o candidato.


## 2. Três pesquisas aleatórias (transformar → conferir)

Sorteie **três** amostras aleatórias de **800 pessoas** (use `eleitorado.sample(800, random_state=1)`, depois 2 e 3). Para cada uma, calcule: proporção estimada, margem de erro (95%), intervalo de confiança e se o intervalo **pegou a verdade**. Monte uma tabela com as três linhas.

Use esta função (já pronta; ela é testada na parte 5):

```python
def margem_erro(p, n, z=1.96):
    return z * np.sqrt(p * (1 - p) / n)
```

In [3]:
def margem_erro(p, n, z=1.96):
    return z * np.sqrt(p * (1 - p) / n)

amostra1 = eleitorado.sample(800, random_state=1)
amostra2 = eleitorado.sample(800, random_state=2)
amostra3 = eleitorado.sample(800, random_state=3)

#proporçao
a1 = amostra1["apoia"].mean()
a2 = amostra2["apoia"].mean()
a3 = amostra3["apoia"].mean()

#margem de erro
m1 = margem_erro(a1, 800)
m2 = margem_erro(a2, 800)
m3 = margem_erro(a3, 800)

tabela = pd.DataFrame({
    "amostra": ["Amostra 1", "Amostra 2", "Amostra 3"],
    "pessoas": [len(amostra1), len(amostra2), len(amostra3)],
    "estimativa": [a1, a2, a3],
    "margem": [m1, m2, m3],
    "de": [a1-m1, a2-m2, a3-m3],
    "ate": [a1+m1, a2+m2, a3+m3],
    "pegou a verdade": [a1-m1 <= verdade <= a1+m1, a2-m2 <= verdade <= a2+m2, a3-m3 <= verdade <= a3+m3],
})

tabela["estimativa"] = (tabela["estimativa"] * 100).round(0).astype(int).astype(str) + "%"
tabela["margem"] = (tabela["margem"] * 100).round(1).astype(str) + "%"
tabela["de"] = (tabela["de"] * 100).round(1).astype(str) + "%"
tabela["ate"] = (tabela["ate"] * 100).round(1).astype(str) + "%"

print(tabela.to_string(index=False))

  amostra  pessoas estimativa margem    de   ate  pegou a verdade
Amostra 1      800        38%   3.4% 35.0% 41.7%             True
Amostra 2      800        39%   3.4% 35.5% 42.3%             True
Amostra 3      800        38%   3.4% 34.5% 41.2%             True


**Pergunta de leitura (2 a 3 linhas):** as três pesquisas deram o mesmo número? Alguma errou além da margem? Isso é esperado? Quantas, em média, errariam se você fizesse 100 pesquisas?

In [ ]:
# Não, as 3 pesquisas não deram o mesmo número, a amostra 1 deu uma estimativa de 38%, a amostra 2 deu uma estimativa de 39% e a amostra 3 deu uma estimativa de 38%, 
# sendo que a estimativa da amostra 2 foi a mais próxima da verdade, que é 39% (sem contar com a margem de erro).
# Nenhuma das amostras errou além da margem de erro, todas pegaram a verdade. As amostras 1 e 3 erraram 1% para baixo, a amostra 2 acertou a estimativa.
# Isso é esperado pois a amostragem é aleatória, então cada amostra pode ter uma composição diferente da população, o que pode levar a estimativas diferentes. 
# Em média, se fossem feitas 100 pesquisas, cerca de 95% delas estariam dentro da margem de erro, e 5% estariam fora.

## 3. A enquete enviesada

Simule uma enquete aberta em que a chance de responder é de **8% para 16-34**, **2% para 35-59** e **0,5% para 60+**. Guarde em `enquete` só quem respondeu. Calcule: número de respostas, estimativa de apoio, a margem "no papel" e o **erro real** (estimativa menos `verdade`). Compare a composição por faixa da enquete com a do eleitorado.

Dica de estrutura (preencha as lacunas):

```python
prob = eleitorado["faixa"].map({"16-34": 0.08, "35-59": 0.02, "60+": 0.005}).to_numpy()
respondeu = rng.random(N) < prob
enquete = eleitorado[respondeu]
```

In [5]:
# SEU CÓDIGO AQUI
aleatoria = eleitorado.sample(800, random_state=1)

prob = eleitorado["faixa"].map({"16-34": 0.08, "35-59": 0.02, "60+": 0.005}).to_numpy()
respondeu = rng.random(N) < prob
enquete = eleitorado[respondeu]

resumo = pd.DataFrame({
    "método": ["Pesquisa aleatória", "Enquete aberta"],
    "respostas": [len(aleatoria), len(enquete)],
    "estimativa": [aleatoria["apoia"].mean(), enquete["apoia"].mean()],
    "margem 'no papel'": [margem_erro(aleatoria["apoia"].mean(), len(aleatoria)),
                          margem_erro(enquete["apoia"].mean(), len(enquete))],
})
resumo["erro real"] = resumo["estimativa"] - verdade
for col in ["estimativa", "margem 'no papel'", "erro real"]:
    resumo[col] = (resumo[col] * 100).round(1)
print(resumo.to_string(index=False))
print()
print("Quem entrou em cada amostra (% por faixa):")
print(pd.DataFrame({
    "eleitorado": eleitorado["faixa"].value_counts(normalize=True),
    "aleatória": aleatoria["faixa"].value_counts(normalize=True),
    "enquete": enquete["faixa"].value_counts(normalize=True),
}).round(3).mul(100).sort_index())

            método  respostas  estimativa  margem 'no papel'  erro real
Pesquisa aleatória        800        38.4                3.4       -0.9
    Enquete aberta       7287        34.0                1.1       -5.3

Quem entrou em cada amostra (% por faixa):
       eleitorado  aleatória  enquete
faixa                                
16-34        34.0       31.0     73.7
35-59        44.1       44.5     22.9
60+          21.9       24.5      3.4


**Pergunta de leitura (3 a 4 linhas):** a enquete tem mais respostas que a pesquisa aleatória. Por que mesmo assim ela erra mais? O que a margem "no papel" da enquete significa, se significa alguma coisa?

In [ ]:
# Ela erra mais, pois a amostra da enquete aberta não é representativa da população, enquanto a amostra aleatória é representativa. 
# Isso acontece por conta da seleção dos públicos que existem ao realizar a enquete. 
# Na tabela, é possível observar que a maior parte de pessoas atingidas pela enquete estão na faixa de 16-34 anos (73.7), enquanto o eleitorado real representa apenas 34% de pessoas nessa faixa.
# A margem 'no papel' da enquete supõe que a amostra é representativa, ou seja, que houve um "sorteio aleatório" das pessoas que responderam ela. 
# Nesse caso, ela não significa nada, por justamente não ser uma pesquisa que representa a população, ou seja, não mede o erro real da enquete.

## 4. Ponderação da enquete

Calcule o peso de cada faixa (`% no eleitorado ÷ % na enquete`), aplique aos respondentes e recalcule o apoio. Compare as três estimativas: **verdade, enquete bruta, enquete ponderada**.

Nesta simulação, quem responde **dentro** de cada faixa é aleatório, e por isso a ponderação funciona bem. **Pense e escreva (3 a 4 linhas):** na vida real, o que poderia fazer a ponderação **não** resolver? Dê um exemplo concreto de uma pesquisa eleitoral.

In [ ]:
# SEU CÓDIGO AQUI
pop_prop = eleitorado["faixa"].value_counts(normalize=True)            
print("Composição do eleitorado:", pop_prop.round(2).to_dict())

amostra = enquete 

amostra_prop = amostra["faixa"].value_counts(normalize=True)
print("Composição da amostra:", amostra_prop.round(2).to_dict())

amostra["peso"] = amostra["faixa"].map(pop_prop / amostra_prop)
print("Pesos:", amostra.groupby("faixa")["peso"].first().round(2).to_dict())

sem_peso = amostra["apoia"].mean()
com_peso = (amostra["apoia"] * amostra["peso"]).sum() / amostra["peso"].sum()
print()
print(f"Verdade do eleitorado (censo):  {verdade:.0%}")
print(f"Enquete sem ponderar:     {sem_peso:.0%}")
print(f"Enquete ponderada:        {com_peso:.0%}")


# Na vida real, o que poderia fazer a ponderação não resolver é justamente que ela só corrige a proporção das faixas, não quem respondeu dentro delas.
# Por exemplo, se existe uma enquete no Instagram sobre que canditado será eleito, as pessoas que votam são as usuárias que buscam por política dentro do Instagram 
# e que seguem a conta que postou a enquete, ou seja, podem não representar a população de eleitores, mesmo com o peso certo. 

Composição do eleitorado: {'35-59': 0.44, '16-34': 0.34, '60+': 0.22}
Composição da amostra: {'16-34': 0.74, '35-59': 0.23, '60+': 0.03}
Pesos: {'16-34': 0.46, '35-59': 1.93, '60+': 6.38}

Verdade do eleitorado (censo):  39%
Enquete sem ponderar:     34%
Enquete ponderada:        40%


## 5. Função pequena: margem de erro como texto

Escreva uma função que recebe a proporção e o tamanho da amostra e devolve um texto pronto para uma matéria.

**Assinatura:** `def frase_pesquisa(p, n):`
**Entrada:** `p` (proporção entre 0 e 1), `n` (inteiro, número de entrevistas).
**Saída:** texto no formato `"41% (±3 p.p., 95% de confiança, 1.000 entrevistas)"`, com a margem arredondada para número inteiro e o `n` com ponto de milhar.
**Exemplo:** `frase_pesquisa(0.41, 1000)` devolve `"41% (±3 p.p., 95% de confiança, 1.000 entrevistas)"`.

Quando o código estiver pronto, rode os testes abaixo.

In [ ]:
def frase_pesquisa(p, n):
    porcentagem = str(round(p * 100)) + "%"
    margem = margem_erro(p, n) * 100
    margem = (round(margem)) 

    n_texto = str(n)
    if n >= 1000:
        n_texto = f"{n:,}".replace(",", ".")

    frase = f"{porcentagem} (±{margem} p.p., 95% de confiança, {n_texto} entrevistas)"
    return frase

assert frase_pesquisa(0.41, 1000) == "41% (±3 p.p., 95% de confiança, 1.000 entrevistas)"
assert frase_pesquisa(0.50, 400) == "50% (±5 p.p., 95% de confiança, 400 entrevistas)"
assert frase_pesquisa(0.38, 2000) == "38% (±2 p.p., 95% de confiança, 2.000 entrevistas)"
print("Testes passaram.")

# IA utilizada para gerar os códigos antes de "assert frase_pesquisa". Claude AI

Testes passaram.


## 6. Uma ficha técnica real (comunicar)

Abra o **PesqEle** (`pesqele-divulgacao.tse.jus.br`) e escolha **uma pesquisa registrada** de 2026. Preencha a tabela abaixo (em Markdown, nesta célula) com os dados reais e o link/número de registro. Depois escreva um **parecer de 5 a 8 linhas**: a manchete que circulou sobre essa pesquisa (se achar) é sustentada pela ficha? A diferença entre os primeiros colocados é maior que a margem **da diferença**? O que você checaria antes de repetir o número?

| Campo | Valor |
|---|---|
| Registro | BR-03663/2026 (Eleições Gerais 2026, cargo: Presidente). Link: https://pesqele-divulgacao.tse.jus.br/app/pesquisa/detalhar.xhtml|
| Contratante / instituto | ATLASINTEL TECNOLOGIA DE DADOS LTDA / ATLASINTEL |
| Período de campo | 03/10/2026 a 08/10/2026|
| Universo | Eleitorado brasileiro |
| Amostra (n) | 5000 entrevistados |
| Plano amostral e coleta | Amostra aleatória pós-estratificada via algortimo interativo com quotas proporcionais. Coleta por questionário estruturado na Web |
| Ponderação | Sim, por sexo, faixa-etária, escolaridade e renda familia, aceitando desvios de até 1 p.p. por variável.|
| Margem de erro / confiança | 1 p.p. pra cima ou para baixo, 95% de confiança|

**Parecer:** A manchete da Gazeta do Povo (09/10/2026, link: https://www.gazetadopovo.com.br/eleicoes/2026/pesquisa-eleitoral-2026/atlasintel-presidente-outubro-2026-segundo-turno/) diz que Flávio Bolsonaro lidera o segundo turno, com 51,1% contra 45,7% de Lula (dos votos totais, contando brancos, nulos e não sabe). A margem da diferença com 5000 entrevistas é de cerca de 2,7 pontos para cima ou para baixo, e a diferença de 5,4 pontos é maior do que isso, ou seja, só pelo erro amostral, a liderança de Flávio se sustenta. Porém, essa margem só mede a "sorte" do sorteio, e a coleta foi por questionário na web, alcançando só quem tem internet e aceitou responder, isto é, a ponderação por sexo, idade, escolaridade e renda não corrige esse viés. Algo importante de se destacar é que na matéria é divulgado que foram 5026 entrevistados e no registro do TSE é informado que foram 5000. Antes de repetir o número eu compararia com pesquisas de outros institutos e com outros métodos durante o mesmo período (03/10/2026 até 08/10/2026). Pode-se dizer que, pela ficha, a manchete se sustenta nos números, mas com uma ressalva de que a liderança de Flávio vale considerando apenas erro amostral, não garante que a coleta online represente o eleitorado.

## Checklist de entrega

- [ ] `semente` trocada pela sua; notebook executado de cima para baixo sem erro.
- [ ] Partes 1 a 4 com código e as respostas de leitura escritas.
- [ ] Parte 5 com os três testes passando.
- [ ] Parte 6 com ficha real (registro anotado) e parecer.
- [ ] `README.md` com instruções, fontes, limitações e declaração de uso de IA.
- [ ] Pelo menos um commit de progresso e um de entrega.